# 02 Correction Validation

This notebook validates `m8_xgb` and the deterministic `m7_dtr` benchmark on Alpha leave-one-station-out folds and Beta transfer. Full model training remains config-controlled.


## 1. Imports And Paths

Load config and show whether this run is smoke-only or full correction validation.


In [ ]:
from pathlib import Path
import sys

# Keep notebook imports stable whether the notebook is run from JupyterLab,
# VS Code, or the repository root.
article_root = Path.cwd()
while article_root.name != "2_journal_article":
    if article_root.parent == article_root:
        raise RuntimeError("Could not locate publication/2_journal_article")
    article_root = article_root.parent
notebook_dir = article_root / "notebooks"
if str(notebook_dir) not in sys.path:
    sys.path.insert(0, str(notebook_dir))

import _experiment_helpers as h

cfg = h.load_config(article_root)
paths = h.article_paths(article_root, cfg)
h.ensure_output_dirs(paths)
print(f"Article root: {article_root}")
print(f"Config schema: {cfg['schema_version']}")
print(f"Output root: {paths.outputs}")

print(f"Full correction validation: {cfg['execution']['run_full_correction_validation']}")


## 2. Confirm Inputs And Folds

The fold sites are recomputed from Alpha labels so the notebook does not depend on hard-coded station choices.


In [ ]:
alpha = h.load_dataset(article_root, cfg, "alpha")
beta = h.load_dataset(article_root, cfg, "beta")
alpha_sites = h.alpha_loso_sites(alpha, cfg)
print(f"Alpha LOSO sites: {alpha_sites}")
print(f"Beta rows: {len(beta):,}")


## 3. Run Correction Validation

In smoke mode this writes the planned folds and placeholder metric rows. In full mode it writes prediction audit CSVs, metrics, curated tables, and figures.


In [ ]:
result = h.run_correction_validation(article_root)
print(result["status"])
result["metrics"].head(20)


## 4. Inspect Beta Transfer Rows

These rows are the main external-validation signal once full validation is enabled.


In [ ]:
metrics = result["metrics"]
metrics.loc[metrics["dataset"] == "Beta"].head(20)
